## Parte 1 — Softmax con temperatura

La softmax con temperatura se define como p_i = exp(z_i/T) / Σ_j exp(z_j/T). La implementación en NumPy es numéricamente estable porque resta el máximo de los logits antes de exponenciar (p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T), con m = max(z)), lo que evita desbordamientos sin alterar el resultado. La entropía se calcula en bits como H = −Σ_i p_i·log2(p_i).

Con los logits dados (t0 = 2.0, t1 = 1.0, t2 = 0.5, t3 = 0.2, t4 = −1.0, t5 = −3.0), las tres distribuciones (cada una suma 1.0) y sus entropías son:

| Token | Logit | T = 0.5 | T = 1.0 | T = 2.0 |
|---|---|---|---|---|
| t0 | 2.0 | 0.8231 | 0.5516 | 0.3583 |
| t1 | 1.0 | 0.1114 | 0.2029 | 0.2173 |
| t2 | 0.5 | 0.0410 | 0.1231 | 0.1693 |
| t3 | 0.2 | 0.0225 | 0.0912 | 0.1457 |
| t4 | −1.0 | 0.0020 | 0.0275 | 0.0800 |
| t5 | −3.0 | 0.0000 | 0.0037 | 0.0294 |
| **Entropía (bits)** | — | **0.9147** | **1.7998** | **2.2888** |

Se observa el efecto esperado de la temperatura: al bajarla (T = 0.5) la distribución se afila y la masa se concentra en t0 (0.8231), mientras que al subirla (T = 2) se aplana y t5 pasa de 0.0000 a 0.0294. La entropía crece monótonamente con T: 0.9147 → 1.7998 → 2.2888 bits.

In [1]:
# T1: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
SUBTAREA T1 — Softmax con temperatura (NumPy, estable) + entropía en bits.
Tarea B — Temperatura, top-p y entropía de la distribución de salida (MMIA 6013).

Calcula la distribución softmax con temperatura para T = 0.5, 1 y 2 con los
logits t0..t5 = 2.0, 1.0, 0.5, 0.2, -1.0, -3.0, la entropía en bits de cada
una, y presenta todo en una tabla con cuatro decimales.
"""

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# Datos del enunciado
# ----------------------------------------------------------------------------
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
TEMPERATURAS = [0.5, 1.0, 2.0]


# ----------------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable (max-subtraction)
# ----------------------------------------------------------------------------
def softmax_con_temperatura(logits, T):
    """p_i = exp(z_i / T) / sum_j exp(z_j / T), restando el máximo de los
    logits escalados para evitar overflow/underflow."""
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - np.max(z)          # estabilidad numérica: max-subtraction
    e = np.exp(z)
    return e / np.sum(e)


def entropia_bits(p):
    """H(p) = -sum p_i * log2(p_i), en bits (términos nulos se omiten)."""
    p = np.asarray(p, dtype=float)
    mask = p > 0
    return float(-np.sum(p[mask] * np.log2(p[mask])))


# ----------------------------------------------------------------------------
# Cálculo de distribuciones y entropías
# ----------------------------------------------------------------------------
distribuciones = {}
entropias = {}
sumas = {}

for T in TEMPERATURAS:
    p = softmax_con_temperatura(LOGITS, T)
    distribuciones[T] = p
    entropias[T] = entropia_bits(p)
    sumas[T] = float(np.sum(p))
    assert abs(sumas[T] - 1.0) < 1e-12, f"La distribución con T={T} no suma 1.0"

# ----------------------------------------------------------------------------
# Tabla con cuatro decimales (distribuciones + entropías)
# ----------------------------------------------------------------------------
tabla = pd.DataFrame(
    {f"T={T}": [f"{p:.4f}" for p in distribuciones[T]] + [f"{entropias[T]:.4f}"]
     for T in TEMPERATURAS},
    index=TOKENS + ["Entropia_bits"],
)
tabla.index.name = "Token"

# ----------------------------------------------------------------------------
# resultados.json (contrato): valores sin redondear + tabla a 4 decimales
# ----------------------------------------------------------------------------
resultados = {
    "subtarea": "T1 - Softmax con temperatura y entropia en bits",
    "logits": {t: float(z) for t, z in zip(TOKENS, LOGITS)},
    "temperaturas": [float(T) for T in TEMPERATURAS],
    "softmax_con_temperatura": {
        f"T={T}": {
            "distribucion": {t: float(p) for t, p in zip(TOKENS, distribuciones[T])},
            "suma_distribucion": sumas[T],
            "entropia_bits": entropias[T],
        }
        for T in TEMPERATURAS
    },
    "entropias_bits": {f"T={T}": entropias[T] for T in TEMPERATURAS},
    "tabla_4_decimales": {
        str(idx): {col: tabla.loc[idx, col] for col in tabla.columns}
        for idx in tabla.index
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------------
# Figura: distribuciones softmax para las tres temperaturas
# ----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(TOKENS))
width = 0.25
for i, T in enumerate(TEMPERATURAS):
    ax.bar(x + (i - 1) * width, distribuciones[T], width, label=f"T = {T}")
ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_ylabel("Probabilidad")
ax.set_xlabel("Token")
ax.set_title("Softmax con temperatura (logits t0..t5)")
ax.legend()
fig.tight_layout()
plt.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------------
# Resumen
# ----------------------------------------------------------------------------
print("Softmax con temperatura (distribuciones y entropia en bits):")
print(tabla.to_string())
print()
for T in TEMPERATURAS:
    print(f"T={T}: suma={sumas[T]:.6f}  entropia={entropias[T]:.4f} bits")
print("\nArchivos generados: resultados.json, t1_softmax_temperatura.png")


Softmax con temperatura (distribuciones y entropia en bits):
                T=0.5   T=1.0   T=2.0
Token                                
t0             0.8231  0.5516  0.3583
t1             0.1114  0.2029  0.2173
t2             0.0410  0.1231  0.1693
t3             0.0225  0.0912  0.1457
t4             0.0020  0.0275  0.0800
t5             0.0000  0.0037  0.0294
Entropia_bits  0.9147  1.7998  2.2888

T=0.5: suma=1.000000  entropia=0.9147 bits
T=1.0: suma=1.000000  entropia=1.7998 bits
T=2.0: suma=1.000000  entropia=2.2888 bits

Archivos generados: resultados.json, t1_softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1, se ordenan los tokens por probabilidad descendente (t0, t1, t2, t3, t4, t5) y se acumula: 0.5516, 0.7546, 0.8776, 0.9688, 0.9963, 1.0000. Con p = 0.9, el conjunto mínimo que alcanza el umbral requiere incluir a t3: tras t2 la acumulada es 0.8776 (< 0.9) y tras t3 es 0.9688 (≥ 0.9). Sobreviven **t0, t1, t2 y t3** (4 tokens, masa del núcleo 0.9688) y se descartan t4 y t5. La distribución renormalizada sobre el núcleo suma 1.0:

| Token | p (T = 1) | Acumulada | ¿En el núcleo? | p renormalizada |
|---|---|---|---|---|
| t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | no | — |
| t5 | 0.0037 | 1.0000 | no | — |

El recorte de la cola reduce la incertidumbre: la entropía de la distribución renormalizada es 1.6341 bits, frente a 1.7998 bits de la distribución completa de T = 1.

In [2]:
# T2: código aprobado por el crítico
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T=1 (salida de T1).

Pasos:
1) Carga la distribución de T=1 desde entrada/T1/resultados.json (contrato de T1).
2) Ordena los tokens por probabilidad descendente.
3) Conserva el conjunto MÍNIMO cuya probabilidad acumulada alcanza p = 0.9.
4) Renormaliza las probabilidades de los tokens supervivientes (suma = 1.0).
5) Escribe resultados.json y guarda la figura T2_top_p_nucleo.png.
"""

import json
import os

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

P = 0.9
RUTA_T1 = os.path.join("entrada", "T1", "resultados.json")
ARCHIVO_SALIDA = "resultados.json"
ARCHIVO_FIGURA = "T2_top_p_nucleo.png"


def softmax_con_temperatura(logits, T):
    """Softmax numéricamente estable con temperatura (respaldo si no está la salida de T1)."""
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def cargar_distribucion_T1():
    """Lee la distribución de T=1 de la salida de T1; si no existe, la recalcula."""
    if os.path.exists(RUTA_T1):
        with open(RUTA_T1, "r", encoding="utf-8") as f:
            res_t1 = json.load(f)
        bloque = res_t1.get("softmax_con_temperatura", {})
        if "T=1.0" in bloque:
            dist = bloque["T=1.0"]["distribucion"]
        elif "T=1" in bloque:
            dist = bloque["T=1"]["distribucion"]
        else:
            raise KeyError("No se encontró la distribución de T=1 en entrada/T1/resultados.json")
        tokens = sorted(dist.keys())  # t0..t5 en orden canónico
        probs = np.array([dist[t] for t in tokens], dtype=float)
        return tokens, probs, "entrada/T1/resultados.json (salida de T1)"
    # Respaldo: recalcular con la softmax con temperatura de la Parte 1
    tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])
    return tokens, softmax_con_temperatura(logits, 1.0), "recalculada con softmax estable (T=1)"


def main():
    # --- 1) Distribución de T=1 proveniente de T1 -----------------------------
    tokens, probs, origen = cargar_distribucion_T1()

    # --- 2) Orden descendente por probabilidad --------------------------------
    orden = np.argsort(-probs, kind="stable")
    tokens_ord = [tokens[i] for i in orden]
    probs_ord = probs[orden]
    acumulada = np.cumsum(probs_ord)

    # --- 3) Conjunto mínimo con probabilidad acumulada >= p -------------------
    if acumulada[-1] < P:  # salvaguarda numérica: si nada alcanza p, se queda todo
        idx = len(acumulada) - 1
    else:
        # primer índice i con acumulada[i] >= p; el núcleo son los índices 0..i
        idx = int(np.searchsorted(acumulada, P, side="left"))
    k = idx + 1

    tokens_nucleo = tokens_ord[:k]
    probs_nucleo = probs_ord[:k]
    acumulada_nucleo = float(acumulada[idx])

    # --- 4) Renormalización ----------------------------------------------------
    q_nucleo = probs_nucleo / probs_nucleo.sum()
    suma_renorm = float(q_nucleo.sum())
    dist_renorm = {t: float(q) for t, q in zip(tokens_nucleo, q_nucleo)}
    tokens_descartados = tokens_ord[k:]

    # Entropía en bits de la distribución renormalizada (informativo)
    mask = q_nucleo > 0
    entropia_bits_renorm = float(-np.sum(q_nucleo[mask] * np.log2(q_nucleo[mask])))

    # Tabla completa (orden descendente, valores sin redondear)
    filas = []
    for i, t in enumerate(tokens_ord):
        filas.append({
            "token": t,
            "prob_T1": float(probs_ord[i]),
            "prob_acumulada": float(acumulada[i]),
            "en_nucleo": bool(i < k),
            "prob_renormalizada": float(q_nucleo[i]) if i < k else 0.0,
        })

    # Tabla de presentación con 4 decimales
    tabla_4d = {
        "token": tokens_ord,
        "prob_T1": [f"{v:.4f}" for v in probs_ord],
        "prob_acumulada": [f"{v:.4f}" for v in acumulada],
        "en_nucleo": ["si" if i < k else "no" for i in range(len(tokens_ord))],
        "prob_renormalizada": [f"{q_nucleo[i]:.4f}" if i < k else "-" for i in range(len(tokens_ord))],
    }

    # --- 5) resultados.json (contrato de la subtarea) --------------------------
    resultados = {
        "subtarea": "T2 - Muestreo de nucleo top-p (p=0.9) sobre la distribucion de T=1",
        "p": P,
        "fuente_distribucion_T1": origen,
        "distribucion_T1": {t: float(p) for t, p in zip(tokens, probs)},
        "orden_descendente": tokens_ord,
        "probabilidades_ordenadas": {t: float(p) for t, p in zip(tokens_ord, probs_ord)},
        "probabilidad_acumulada_ordenada": {t: float(a) for t, a in zip(tokens_ord, acumulada)},
        "tokens_supervivientes": tokens_nucleo,
        "num_tokens_supervivientes": k,
        "probabilidad_acumulada_nucleo": acumulada_nucleo,
        "distribucion_renormalizada": dist_renorm,
        "suma_distribucion_renormalizada": suma_renorm,
        "tokens_descartados": tokens_descartados,
        "entropia_bits_renormalizada": entropia_bits_renorm,
        "tabla_completa": filas,
        "tabla_4_decimales": tabla_4d,
    }
    with open(ARCHIVO_SALIDA, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

    # --- Figura -----------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    ax = axes[0]
    colores = ["#2a9d8f" if i < k else "#e76f51" for i in range(len(tokens_ord))]
    ax.bar(tokens_ord, probs_ord, color=colores, edgecolor="black", linewidth=0.5)
    ax.set_ylabel("P(token | T=1)")
    ax.set_ylim(0, max(probs_ord) * 1.15)
    ax2 = ax.twinx()
    ax2.plot(tokens_ord, acumulada, "o-", color="#264653", label="acumulada")
    ax2.axhline(P, color="gray", linestyle="--", linewidth=1, label=f"p = {P}")
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel("Probabilidad acumulada")
    ax2.legend(loc="lower right", fontsize=8)
    ax.set_title("Distribución T=1 ordenada + acumulada\n(verde = núcleo top-p, rojo = descartado)")

    ax = axes[1]
    ax.bar(tokens_nucleo, q_nucleo, color="#2a9d8f", edgecolor="black", linewidth=0.5)
    for i, q in enumerate(q_nucleo):
        ax.text(i, q + 0.01, f"{q:.4f}", ha="center", fontsize=9)
    ax.set_ylim(0, max(q_nucleo) * 1.18)
    ax.set_ylabel("Probabilidad renormalizada")
    ax.set_title(f"Distribución renormalizada del núcleo\n{tokens_nucleo} (suma = {suma_renorm:.6f})")

    fig.tight_layout()
    fig.savefig(ARCHIVO_FIGURA, dpi=120)
    plt.close(fig)

    # --- Resumen ------------------------------------------------------------------
    print("=" * 66)
    print("T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T=1")
    print("=" * 66)
    print(f"Fuente de la distribución T=1: {origen}")
    print("Distribución T=1 (orden descendente):")
    for t, p_, a in zip(tokens_ord, probs_ord, acumulada):
        marca = "  <- núcleo" if t in tokens_nucleo else ""
        print(f"  {t}: p = {p_:.6f}   acumulada = {a:.6f}{marca}")
    print(f"\nTokens supervivientes (conjunto mínimo con acumulada >= {P}): {tokens_nucleo}")
    print(f"Probabilidad acumulada del núcleo: {acumulada_nucleo:.6f}")
    print("Distribución renormalizada:")
    for t, q in dist_renorm.items():
        print(f"  {t}: {q:.6f}")
    print(f"Suma de la distribución renormalizada: {suma_renorm:.6f}")
    print(f"Tokens descartados: {tokens_descartados}")
    print(f"Entropía (bits) de la distribución renormalizada: {entropia_bits_renorm:.6f}")
    print(f"\nArchivos escritos: {ARCHIVO_SALIDA}, {ARCHIVO_FIGURA}")


if __name__ == "__main__":
    main()


T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T=1
Fuente de la distribución T=1: entrada/T1/resultados.json (salida de T1)
Distribución T=1 (orden descendente):
  t0: p = 0.551623   acumulada = 0.551623  <- núcleo
  t1: p = 0.202931   acumulada = 0.754553  <- núcleo
  t2: p = 0.123084   acumulada = 0.877637  <- núcleo
  t3: p = 0.091183   acumulada = 0.968820  <- núcleo
  t4: p = 0.027464   acumulada = 0.996283
  t5: p = 0.003717   acumulada = 1.000000

Tokens supervivientes (conjunto mínimo con acumulada >= 0.9): ['t0', 't1', 't2', 't3']
Probabilidad acumulada del núcleo: 0.968820
Distribución renormalizada:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
Suma de la distribución renormalizada: 1.000000
Tokens descartados: ['t4', 't5']
Entropía (bits) de la distribución renormalizada: 1.634075

Archivos escritos: resultados.json, T2_top_p_nucleo.png


## Parte 3 — Comprobación empírica

Se tomaron 10 000 muestras de la distribución renormalizada de la Parte 2 con `numpy.random.default_rng(0)`. Las frecuencias observadas coinciden con las probabilidades teóricas dentro del ruido de muestreo (diferencias de a lo sumo 0.0026):

| Token | Prob. teórica | Conteo observado | Frecuencia observada | Diferencia (obs − teo) |
|---|---|---|---|---|
| t0 | 0.5694 | 5682 | 0.5682 | −0.0012 |
| t1 | 0.2095 | 2093 | 0.2093 | −0.0002 |
| t2 | 0.1270 | 1296 | 0.1296 | +0.0026 |
| t3 | 0.0941 | 929 | 0.0929 | −0.0012 |
| t4 | 0.0000 | 0 | 0.0000 | 0.0000 |
| t5 | 0.0000 | 0 | 0.0000 | 0.0000 |

La divergencia KL de la empírica respecto de la teórica, KL(P_emp ‖ P_teo) = Σ_i p_emp,i·log2(p_emp,i / p_teo,i), es de **5.0069e-05 bits** (equivalente a 3.4705e-05 nats). Los términos con p_emp = 0 contribuyen 0, y t4 y t5, con probabilidad teórica 0, no recibieron ninguna muestra. Un KL tan próximo a cero confirma que el generador reproduce la distribución objetivo; el gráfico de barras generado en la celda muestra las frecuencias observadas prácticamente indistinguibles de las probabilidades teóricas en cada token.

In [3]:
# T3: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
SUBTAREA T3: Comprobación empírica de la distribución renormalizada de T2.
- Toma 10,000 muestras con numpy.random.default_rng(0).
- Grafica en barras las frecuencias observadas frente a las probabilidades teóricas.
- Calcula la divergencia KL (en bits) de la distribución empírica respecto de la teórica.
"""

import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------- 1. Cargar la distribución renormalizada de T2 ----------
with open("entrada/T2/resultados.json", "r", encoding="utf-8") as f:
    t2 = json.load(f)

tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
renorm = t2["distribucion_renormalizada"]  # contiene solo los tokens supervivientes
p_teo = np.array([renorm.get(tok, 0.0) for tok in tokens], dtype=float)

assert abs(p_teo.sum() - 1.0) < 1e-9, "La distribución teórica debe sumar 1"

# ---------- 2. Muestreo: 10,000 muestras con default_rng(0) ----------
N = 10_000
rng = np.random.default_rng(0)
idx = np.arange(len(tokens))
muestras = rng.choice(idx, size=N, p=p_teo)

conteos = np.bincount(muestras, minlength=len(tokens))
f_obs = conteos / N

# ---------- 3. Divergencia KL (bits): KL(P_emp || P_teo) ----------
# KL = sum_i p_emp_i * log2(p_emp_i / p_teo_i)
# Los términos con p_emp_i = 0 contribuyen 0 (límite x*log(x)->0).
# p_teo_i = 0 (t4, t5) solo ocurre donde p_emp_i = 0 (nunca se muestrean),
# por lo que la divergencia está bien definida.
mask = f_obs > 0
kl_bits = float(np.sum(f_obs[mask] * np.log2(f_obs[mask] / p_teo[mask])))
kl_nats = float(np.sum(f_obs[mask] * np.log(f_obs[mask] / p_teo[mask])))

# ---------- 4. Gráfico de barras: observado vs teórico ----------
x = np.arange(len(tokens))
width = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - width / 2, f_obs, width,
            label="Frecuencia observada (10,000 muestras)",
            color="#4C72B0", edgecolor="black", linewidth=0.5)
b2 = ax.bar(x + width / 2, p_teo, width,
            label="Probabilidad teórica (top-p p=0.9 renormalizada)",
            color="#DD8452", edgecolor="black", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_title("Frecuencias observadas vs probabilidades teóricas\n"
             "Distribución renormalizada top-p (T=1, p=0.9); "
             "10,000 muestras, default_rng(0)")
ax.legend(loc="upper right")
for bars in (b1, b2):
    for rect in bars:
        h = rect.get_height()
        ax.annotate(f"{h:.4f}",
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)
ax.set_ylim(0, max(f_obs.max(), p_teo.max()) * 1.18)
plt.tight_layout()
plt.savefig("T3_frecuencias_vs_teoricas.png", dpi=120)
plt.close(fig)

# ---------- 5. Guardar resultados.json (contrato de la subtarea) ----------
resultados = {
    "subtarea": ("T3 - Comprobacion empirica: 10,000 muestras de la distribucion "
                 "renormalizada de T2, comparacion de frecuencias observadas vs "
                 "probabilidades teoricas y divergencia KL en bits"),
    "generador": "numpy.random.default_rng(0)",
    "n_muestras": N,
    "fuente_distribucion_teorica": "entrada/T2/resultados.json (distribucion_renormalizada)",
    "distribucion_teorica_renormalizada": {tok: float(p) for tok, p in zip(tokens, p_teo)},
    "conteos_observados": {tok: int(c) for tok, c in zip(tokens, conteos)},
    "frecuencias_observadas": {tok: float(fr) for tok, fr in zip(tokens, f_obs)},
    "comparacion_por_token": [
        {"token": tok,
         "frecuencia_observada": float(fr),
         "probabilidad_teorica": float(p),
         "diferencia_obs_menos_teo": float(fr - p)}
        for tok, fr, p in zip(tokens, f_obs, p_teo)
    ],
    "kl_bits_empirica_respecto_teorica": kl_bits,
    "kl_nats_empirica_respecto_teorica": kl_nats,
    "definicion_kl": ("KL(P_emp || P_teo) = sum_i p_emp_i * log2(p_emp_i / p_teo_i); "
                      "los terminos con p_emp_i = 0 contribuyen 0. Los tokens t4 y t5 "
                      "tienen probabilidad teorica 0 y recibieron 0 muestras."),
    "figura": "T3_frecuencias_vs_teoricas.png",
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ---------- 6. Resumen ----------
print("=== T3: Comprobacion empirica ===")
print(f"Muestras: {N} con numpy.random.default_rng(0)")
print(f"{'token':<6}{'conteo':>10}{'observada':>12}{'teorica':>12}")
for tok, c, fr, p in zip(tokens, conteos, f_obs, p_teo):
    print(f"{tok:<6}{int(c):>10d}{fr:>12.6f}{p:>12.6f}")
print(f"Divergencia KL (empirica || teorica) = {kl_bits:.6f} bits "
      f"({kl_nats:.6f} nats)")
print("Figura guardada: T3_frecuencias_vs_teoricas.png")
print("Resultados guardados en resultados.json")


=== T3: Comprobacion empirica ===
Muestras: 10000 con numpy.random.default_rng(0)
token     conteo   observada     teorica
t0          5682    0.568200    0.569376
t1          2093    0.209300    0.209462
t2          1296    0.129600    0.127045
t3           929    0.092900    0.094117
t4             0    0.000000    0.000000
t5             0    0.000000    0.000000
Divergencia KL (empirica || teorica) = 0.000050 bits (0.000035 nats)
Figura guardada: T3_frecuencias_vs_teoricas.png
Resultados guardados en resultados.json


## Parte 4 — Pregunta conceptual

**Cuando T → 0.** La razón entre las probabilidades de dos tokens es p_i/p_j = exp((z_i − z_j)/T); si z_i > z_j, esa razón tiende a infinito cuando T → 0⁺. Toda la masa se concentra entonces en el token de mayor logit (aquí t0) y la entropía tiende a 0 bits. Las entropías medidas en la Parte 1 muestran esa tendencia: al reducir T de 2 a 1 y de 1 a 0.5, la entropía cae de 2.2888 a 1.7998 y a 0.9147 bits, mientras la probabilidad de t0 sube de 0.3583 a 0.5516 y a 0.8231; extrapolando, en el límite la distribución es una masa puntual sobre t0.

**Por qué T → 0 equivale a la decodificación voraz.** La decodificación voraz (greedy) selecciona en cada paso el token de máxima probabilidad, es decir, el argmax de los logits. Si la distribución con T → 0 asigna probabilidad 1 al argmax y 0 al resto, muestrear de ella devuelve ese token de forma determinista: el muestreo y greedy producen exactamente la misma salida. La temperatura muy baja convierte el muestreo estocástico en la elección determinista del token más probable.

**Cuando T → ∞.** Los exponentes z_i/T tienden a 0, de modo que exp(z_i/T) → 1 para todos los tokens y la distribución tiende a la uniforme sobre los seis tokens, cuya entropía es el máximo posible para ese vocabulario: log2(6) bits. Las entropías medidas se aproximan a ese límite desde abajo a medida que T crece: 0.9147 → 1.7998 → 2.2888 bits, todavía por debajo del máximo en T = 2.

En síntesis, la temperatura interpola entre dos extremos cuantificados por la entropía: T → 0 da una distribución determinista (H → 0 bits, equivalente a greedy) y T → ∞ da la distribución uniforme (H → log2(6) bits, máxima aleatoriedad).